In [71]:
import asyncio
import json
import heapq
from dataclasses import dataclass, field
from enum import Enum
from typing import List, Tuple, Dict, Optional
from autogen_core import RoutedAgent, message_handler, SingleThreadedAgentRuntime, AgentId, DefaultTopicId, TypePrefixSubscription
from pydantic import BaseModel

Definición entorno

In [72]:
def generar_y_mostrar_rejilla(datos):
    # 1. Extraer dimensiones del grid
    N = datos["grid"]["N"]
    M = datos["grid"]["M"]
    
    # 2. Crear la rejilla vacía (N filas x M columnas) llenas de "." (libre)
    # Cada celda será una lista para poder meter varios elementos si coinciden
    rejilla = [[["."] for _ in range(M)] for _ in range(N)]
    
    # Función auxiliar para colocar elementos y quitar el "." por defecto
    def colocar(ubicacion, texto):
        fila, col = ubicacion
        if rejilla[fila][col] == ["."]:
            rejilla[fila][col] = [texto]
        else:
            rejilla[fila][col].append(texto)

    # 3. Colocar Obstáculos (#) y Tráfico (T)
    for obs in datos["grid"]["obstaculos"]:
        colocar(obs, "#")
    for tr in datos["grid"]["trafico"]:
        colocar(tr, "T")
        
    # 4. Colocar Depósitos (D1, D2...)
    for deposito in datos["depositos"]:
        colocar(deposito["ubicacion"], deposito["id"])
        
    # 5. Colocar Clientes (C1, C2...)
    for cliente in datos["clientes"]:
        colocar(cliente["ubicacion"], cliente["id"])
        
    # 6. Colocar Vehículos (V1, V2...)
    for vehiculo in datos["vehiculos"]:
        colocar(vehiculo["ubicacion"], vehiculo["id"])
        
    # 7. Imprimir la rejilla por consola de forma bonita
    print("--- MAPA DEL CENTRO LOGÍSTICO ---")
    for fila in rejilla:
        # Formateamos cada celda para que ocupe el mismo ancho (ej: 5 caracteres)
        # uniendo los elementos si hay varios (ej: "D1,V1")
        fila_formateada = [",".join(celda).ljust(5) for celda in fila]
        print(" ".join(fila_formateada))
    print("-" * 33)
    
    return rejilla

In [73]:
# Tu JSON exacto convertido a diccionario de Python
parametros_entorno = {
  "grid": {
    "N": 10, 
    "M": 10,
    "obstaculos": [[1, 2], [1, 3], [2, 2]],
    "trafico": [[4, 5], [4, 6]]
  },
  "depositos": [
    {"id": "D1", "ubicacion": [0, 0], "inventario_total": 35, "paquetes": {"pares": 15, "impares": 5}},
    {"id": "D2", "ubicacion": [9, 0], "inventario_total": 25, "paquetes": {"pares": 10, "impares": 5}}
  ],
  "clientes": [
    {"id": "C1", "ubicacion": [8, 8], "prioridad": "alta", "demanda_por_deposito": {"D1": 15, "D2": 5}},
    {"id": "C2", "ubicacion": [2, 8], "prioridad": "media", "demanda_por_deposito": {"D1": 0, "D2": 10}}
  ],
  "vehiculos": [
    {"id": "V1", "ubicacion": [5, 5], "capacidad": 20, "coste_por_paso": 2},
    {"id": "V2", "ubicacion": [4, 4], "capacidad": 10, "coste_por_paso": 1}
  ]
}

# Ejecutamos la función
matriz_estado = generar_y_mostrar_rejilla(parametros_entorno)

--- MAPA DEL CENTRO LOGÍSTICO ---
D1    .     .     .     .     .     .     .     .     .    
.     .     #     #     .     .     .     .     .     .    
.     .     #     .     .     .     .     .     C2    .    
.     .     .     .     .     .     .     .     .     .    
.     .     .     .     V2    T     T     .     .     .    
.     .     .     .     .     V1    .     .     .     .    
.     .     .     .     .     .     .     .     .     .    
.     .     .     .     .     .     .     .     .     .    
.     .     .     .     .     .     .     .     C1    .    
D2    .     .     .     .     .     .     .     .     .    
---------------------------------


In [74]:
class Deposito(BaseModel):
    id: str
    posicion: Tuple[int, int]
    paquetes_pares: int
    paquetes_impares: int

class Cliente(BaseModel):
    id: str
    posicion: Tuple[int, int]
    prioridad: str
    demandas: Dict[str, int]
    recibido: Dict[str, int] = {}

class VehiculoEntorno(BaseModel):
    id: str
    posicion: Tuple[int, int]
    capacidad_max: int
    carga_actual: int = 0
    coste_paso: int=1
    metricas: Dict[str, int] = {
        "desplazamientos": 0,
        "coste_total": 0,
        "espacio_desperdiciado": 0,
        "capacidad_ociosa": 0
    }

class EstadoEntorno(BaseModel):
    mapa: List[List[str]]
    depositos: List[Deposito]
    clientes: List[Cliente]
    vehiculos: List[VehiculoEntorno]
    dimensiones: Tuple[int, int]
    penalizacion_trafico: int = 3

In [75]:
import heapq
from typing import List, Tuple, Optional
from pydantic import BaseModel

class AgenteVehiculoSimple(RoutedAgent):
    def __init__(self, id, capacidad, coste_paso, penalizacion_t, posicion_inicial=(0,0)):
        super().__init__(id)
        self.capacidad_max = capacidad  
        self.coste_paso = coste_paso
        self.posicion = posicion_inicial
        self.carga_actual = 0
        self.plan_ruta = []
        self.penalizacion_t = penalizacion_t
        self.inventario_interno = {}
        
        # NUEVO: Estado explícito de la tarea
        # Formato esperado: {"tipo": "CARGAR"|"ENTREGAR", "cliente_id": "C1", "deposito_id": "D1", "destino": (x,y)}
        self.tarea_actual = None 

        self.metricas = {
            "desplazamientos": 0,
            "coste_total": 0,
            "espacio_desperdiciado": 0,
            "capacidad_ociosa": 0
        }

    @message_handler
    async def actualizar_y_decidir(self, message: EstadoEntorno, ctx) -> None:
        
        # 1. Validar si la tarea actual sigue teniendo sentido en el entorno actual
        if self.tarea_actual:
            if not self._tarea_es_valida(message):
                print(f"⚠️ [{self.id}] Tarea inválida (alguien se adelantó o se vació el stock). Abortando...")
                self.tarea_actual = None
                self.plan_ruta = []

        # 2. Si no tengo tarea (o se acaba de cancelar), busco una nueva
        if not self.tarea_actual:
            self._asignar_nueva_tarea(message)
        
        # 3. Ejecutar un único paso de movimiento si tengo una ruta trazada
        if self.plan_ruta:
            siguiente_pos = self.plan_ruta.pop(0)
            self.posicion = siguiente_pos
            self.metricas["desplazamientos"] += 1

            coste_paso_actual = self.coste_paso
            if message.mapa[self.posicion[0]][self.posicion[1]] == 'T':
                coste_paso_actual += self.penalizacion_t
            self.metricas["coste_total"] += coste_paso_actual
            self.metricas["espacio_desperdiciado"] += (self.capacidad_max - self.carga_actual)

            # Actualizar mi reflejo en el estado global para que otros lo vean (si aplica)
            for v_entorno in message.vehiculos:
                if v_entorno.id == self.id.type:
                    v_entorno.posicion = self.posicion
                    v_entorno.carga_actual = self.carga_actual
                    v_entorno.metricas = self.metricas
                    
            print(f"🚚 [{self.id}] Moviendo a {self.posicion} | Destino final: {self.tarea_actual['destino']}")
            
            # 4. Verificar si he llegado al destino para actuar
            if self.posicion == self.tarea_actual["destino"]:
                self.ejecutar_accion_destino(message)
        else:
            
            self.metricas["capacidad_ociosa"] += self.capacidad_max
            for v_entorno in message.vehiculos:
                if v_entorno.id == self.id.type:
                    v_entorno.metricas = self.metricas
            print(f"💤 [{self.id}] En espera. No hay tareas o no hay ruta.")

    def _tarea_es_valida(self, entorno) -> bool:
        """Comprueba si el entorno sigue permitiendo cumplir la tarea asignada"""
        cliente = next((c for c in entorno.clientes if c.id == self.tarea_actual["cliente_id"]), None)
        
        if not cliente: 
            return False

        if self.tarea_actual["tipo"] == "CARGAR":
            depo_id = self.tarea_actual["deposito_id"]
            depo = next((d for d in entorno.depositos if d.id == depo_id), None)
            
            # ¿Queda stock en el depósito?
            if not depo or (depo.paquetes_pares + depo.paquetes_impares) <= 0:
                return False
            # ¿El cliente sigue necesitando este producto?
            if cliente.demandas.get(depo_id, 0) <= cliente.recibido.get(depo_id, 0):
                return False
                
        elif self.tarea_actual["tipo"] == "ENTREGAR":
            # ¿El cliente sigue necesitando lo que llevo en el camión?
            sigue_necesitando = False
            for d_id, cant in self.inventario_interno.items():
                if cant > 0 and cliente.demandas.get(d_id, 0) > cliente.recibido.get(d_id, 0):
                    sigue_necesitando = True
                    break
            if not sigue_necesitando:
                # El camión se queda con carga "zombi" ocupando espacio
                return False

        return True

    def _asignar_nueva_tarea(self, entorno):
        """Asignación por prioridad de cliente, resolviendo empates por cercanía del depósito"""
        
        # CASO A: Tengo carga en el camión. Prioridad absoluta: ir a entregarla.
        if self.carga_actual > 0:
            for c in entorno.clientes:
                for dep_id, cant in self.inventario_interno.items():
                    if cant > 0 and c.demandas.get(dep_id, 0) > c.recibido.get(dep_id, 0):
                        self.tarea_actual = {
                            "tipo": "ENTREGAR",
                            "cliente_id": c.id,
                            "deposito_id": dep_id,
                            "destino": c.posicion
                        }
                        self.plan_ruta = self.calcular_a_estrella(self.posicion, c.posicion, entorno.mapa)
                        return

        # CASO B: Estoy vacío. Busco a quién servir.
        clientes_pendientes = [c for c in entorno.clientes if sum(c.demandas.values()) > sum(c.recibido.values())]
        if not clientes_pendientes:
            return  # No hay más trabajo global

        # 1. Agrupar clientes por su prioridad numérica
        val_prioridad = {"alta": 3, "media": 2, "baja": 1}
        # Ordenamos descendente para tener los más altos primero
        clientes_pendientes.sort(key=lambda c: val_prioridad.get(c.prioridad, 1), reverse=True)
        
        max_prioridad_actual = val_prioridad.get(clientes_pendientes[0].prioridad, 1)
        
        mejor_tarea = None
        menor_coste = float('inf')

        # 2. Evaluar SOLO a los clientes empatados en la prioridad más alta disponible
        for cliente in clientes_pendientes:
            if val_prioridad.get(cliente.prioridad, 1) < max_prioridad_actual:
                break # Dejamos de mirar, ya pasamos a una prioridad menor
            
            # Para este cliente de alta prioridad, evaluamos qué depósitos necesita
            for dep_id, demanda in cliente.demandas.items():
                pendiente = demanda - cliente.recibido.get(dep_id, 0)
                if pendiente > 0:
                    deposito = next((d for d in entorno.depositos if d.id == dep_id), None)
                    if deposito and (deposito.paquetes_pares > 0 or deposito.paquetes_impares > 0):
                        
                        # Criterio de desempate: el depósito con menor coste de acceso
                        coste, ruta_tentativa = self.estimar_coste_y_ruta(deposito.posicion, entorno.mapa)
                        if coste < menor_coste:
                            menor_coste = coste
                            mejor_ruta = ruta_tentativa
                            mejor_tarea = {
                                "tipo": "CARGAR",
                                "cliente_id": cliente.id,
                                "deposito_id": dep_id,
                                "destino": deposito.posicion
                            }

        # 3. Asignar la mejor tarea encontrada y trazar ruta inicial
        if mejor_tarea:
            self.tarea_actual = mejor_tarea
            self.plan_ruta = self.calcular_a_estrella(self.posicion, mejor_tarea["destino"], entorno.mapa)
            print(f"🎯 [{self.id}] Nueva tarea asignada: {self.tarea_actual['tipo']} en {self.tarea_actual['destino']} para Cliente {self.tarea_actual['cliente_id']}")

    def ejecutar_accion_destino(self, entorno):
        """Lógica de carga o descarga al llegar al destino"""
        
        if self.tarea_actual["tipo"] == "CARGAR":
            depo = next((d for d in entorno.depositos if d.id == self.tarea_actual["deposito_id"]), None)
            if depo:
                espacio = self.capacidad_max - self.carga_actual
                # Lógica de carga: simplificada para el ejemplo
                pares = min(espacio // 2, depo.paquetes_pares)
                impares = min(espacio - (pares * 2), depo.paquetes_impares)
                unidades_cargadas = (pares * 2) + impares
                
                if unidades_cargadas > 0:
                    self.inventario_interno[depo.id] = self.inventario_interno.get(depo.id, 0) + unidades_cargadas
                    self.carga_actual += unidades_cargadas
                    
                    depo.paquetes_pares -= pares
                    depo.paquetes_impares -= impares
                    print(f"📦 [{self.id}] Cargó {unidades_cargadas} uds del depósito {depo.id}.")
            
            # Tras cargar, la tarea se elimina para que en el siguiente turno se genere la tarea de "ENTREGAR"
            self.tarea_actual = None

        elif self.tarea_actual["tipo"] == "ENTREGAR":
            cliente = next((c for c in entorno.clientes if c.id == self.tarea_actual["cliente_id"]), None)
            if cliente:
                for dep_id in list(self.inventario_interno.keys()):
                    cant_camion = self.inventario_interno[dep_id]
                    pendiente = cliente.demandas.get(dep_id, 0) - cliente.recibido.get(dep_id, 0)
                    entrega = min(cant_camion, pendiente)
                    
                    if entrega > 0:
                        cliente.recibido[dep_id] = cliente.recibido.get(dep_id, 0) + entrega
                        self.inventario_interno[dep_id] -= entrega
                        self.carga_actual -= entrega
                        print(f"✅ [{self.id}] Entregó {entrega} uds (Origen: {dep_id}) al cliente {cliente.id}.")
            
            self.tarea_actual = None

    def estimar_coste_y_ruta(self, destino, mapa) -> Tuple[float, List[Tuple[int, int]]]:
        """
        Calcula el coste real de una ruta y devuelve la ruta misma para no recalcular.
        """
        ruta_tentativa = self.calcular_a_estrella(self.posicion, destino, mapa)
        if not ruta_tentativa:
            return float('inf'), []
        
        coste_total = 0
        for paso in ruta_tentativa:
            coste_total += self.coste_paso
            if mapa[paso[0]][paso[1]] == 'T':
                coste_total += self.penalizacion_t [cite: 19]
        return coste_total, ruta_tentativa
    
    def calcular_a_estrella(self, inicio, fin, mapa):
        # 8 direcciones de movimiento 
        direcciones = [(0, 1), (0, -1), (1, 0), (-1, 0), (1, 1), (1, -1), (-1, 1), (-1, -1)]
        open_list = [(0, inicio)]
        camino_previo = {}
        coste_acumulado = {inicio: 0}

        while open_list:
            _, actual = heapq.heappop(open_list)
            if actual == fin: return self.reconstruir_camino(camino_previo, actual)
            
            for dx, dy in direcciones:
                vecino = (actual[0] + dx, actual[1] + dy)
                if self.es_transitable(vecino, mapa):
                    # Coste adicional si es zona de tráfico (T) [cite: 10, 45]
                    coste = self.coste_paso + (self.penalizacion_t if mapa[vecino[0]][vecino[1]] == 'T' else 0)
                    nuevo_coste = coste_acumulado[actual] + coste
                    
                    if vecino not in coste_acumulado or nuevo_coste < coste_acumulado[vecino]:
                        coste_acumulado[vecino] = nuevo_coste
                        prioridad = nuevo_coste + self.distancia_chebyshev(vecino, fin)
                        heapq.heappush(open_list, (prioridad, vecino))
                        camino_previo[vecino] = actual
        return []

    def distancia_chebyshev(self, p1, p2):
        return max(abs(p1[0] - p2[0]), abs(p1[1] - p2[1]))

    def es_transitable(self, pos, mapa):
        # Evita obstáculos (#) 
        return 0 <= pos[0] < len(mapa) and 0 <= pos[1] < len(mapa[0]) and mapa[pos[0]][pos[1]] != '#'

    def reconstruir_camino(self, camino_previo, actual):
        camino = []
        while actual in camino_previo:
            camino.append(actual)
            actual = camino_previo[actual]
        return camino[::-1]

In [76]:
def mostrar_metricas_finales(estado_final, iteraciones_totales):
    
    # Usamos estado_final.vehiculos en lugar de lista_agentes
    total_desplazamientos = sum(v.metricas["desplazamientos"] for v in estado_final.vehiculos)
    total_coste = sum(v.metricas["coste_total"] for v in estado_final.vehiculos)
    total_desperdicio = sum(v.metricas["espacio_desperdiciado"] for v in estado_final.vehiculos)
    total_ociosa = sum(v.metricas["capacidad_ociosa"] for v in estado_final.vehiculos)

    print("\n" + "="*50)
    print("📊 MÉTRICAS FINALES DE LA SIMULACIÓN 📊")
    print("="*50)
    # Tiempo total (número de iteraciones) de la simulación [cite: 54]
    print(f"⏱️  Tiempo total (Iteraciones)    : {iteraciones_totales}")
    # Número de desplazamientos [cite: 55]
    print(f"🚚 Número de desplazamientos     : {total_desplazamientos}")
    # Coste total asociado a los vehículos [cite: 57]
    print(f"💰 Coste total asociado          : {total_coste} uds")
    # Espacio total desperdiciado por los vehículos en cada ruta [cite: 56]
    print(f"💨 Espacio desperdiciado en ruta : {total_desperdicio} uds de volumen")
    # Capacidad total de los vehículos que no están en ruta [cite: 58]
    print(f"💤 Capacidad ociosa (en espera)  : {total_ociosa} uds de volumen")
    print("="*50)

In [77]:
import os

def mostrar_mapa_notebook(estado, iteracion):
    N, M = estado.dimensiones
    print(f"=== 🚚 SIMULACIÓN LOGÍSTICA - ITERACIÓN {iteracion} ===")
    
    grid_visual = [fila[:] for fila in estado.mapa]
    for d in estado.depositos: grid_visual[d.posicion[0]][d.posicion[1]] = f"[{d.id}]"
    for c in estado.clientes: grid_visual[c.posicion[0]][c.posicion[1]] = f"[{c.id}]"
    for v in estado.vehiculos:
        x, y = v.posicion
        if grid_visual[x][y] in ['.', '#', 'T'] or not grid_visual[x][y].startswith('['):
            grid_visual[x][y] = f"V{v.id[-1]}"
        else:
            grid_visual[x][y] += f",V{v.id[-1]}"

    for fila in grid_visual:
        print(" ".join(str(celda).ljust(8) for celda in fila))
        
    # Mostrar estado de los clientes
    print("-" * 40)
    for c in estado.clientes:
        print(f"Cliente {c.id} ({c.prioridad}) -> Demanda: {c.demandas} | Recibido: {c.recibido}")

In [78]:
from autogen_core import AgentId

async def ejecutar_simulacion_simple(runtime, estado_actual):
    iteracion = 0
    simulacion_terminada = False
    
    # Visualización inicial
    mostrar_mapa_notebook(estado_actual, iteracion)
    await asyncio.sleep(1)

    while not simulacion_terminada and iteracion < 100:
        iteracion += 1
        
        # 1. Enviar el estado a cada vehículo a través del Runtime de AutoGen
        for v in estado_actual.vehiculos:
            # Construimos la dirección del agente (su ID y tipo)
            agente_id = AgentId(type=v.id, key="default") 
            
            # Enviamos el mensaje (el framework dispara el @message_handler automáticamente)
            await runtime.send_message(
                message=estado_actual, 
                recipient=agente_id
            )
        
        # 2. Visualizar turno
        mostrar_mapa_notebook(estado_actual, iteracion)
        await asyncio.sleep(0.5)

        # 3. Comprobar finalización (misma lógica de antes)
        mercancia_entregada = True
        for cliente in estado_actual.clientes:
            for dep_id, demanda in cliente.demandas.items():
                if cliente.recibido.get(dep_id, 0) < demanda:
                    mercancia_entregada = False
                    break
        
        if mercancia_entregada:
            simulacion_terminada = True
            print(f"\n🎉 ¡SIMULACIÓN COMPLETADA EN {iteracion} ITERACIONES!")

    # Calcular y mostrar métricas de salida aquí
    mostrar_metricas_finales(estado_actual, iteracion)

In [79]:
N, M = 10, 10
mapa_base = [['.' for _ in range(M)] for _ in range(N)]
for obs in [[1, 2], [1, 3], [2, 2], [3,3],[7,9], [4,4], [6,8]]: mapa_base[obs[0]][obs[1]] = '#'
for tr in [[4, 5], [4, 6], [4,7]]: mapa_base[tr[0]][tr[1]] = 'T'

# Crear el estado inicial
estado_inicial = EstadoEntorno(
    mapa=mapa_base,
    dimensiones=(N, M),
    penalizacion_trafico=3,
    depositos=[
        Deposito(id="D1", posicion=(0, 0), paquetes_pares=30, paquetes_impares=20),
        Deposito(id="D2", posicion=(7, 3), paquetes_pares=40, paquetes_impares=15)
    ],
    clientes=[
        Cliente(id="C1", posicion=(8, 8), prioridad="alta", demandas={"D1": 15, "D2": 25}, recibido={}),
        Cliente(id="C2", posicion=(2, 6), prioridad="media", demandas={"D1": 0, "D2": 10}, recibido={}),
        Cliente(id="C3", posicion=(2, 8), prioridad="media", demandas={"D1": 20, "D2": 15}, recibido={}),
        Cliente(id="C4", posicion=(4, 9), prioridad="alta", demandas={"D1": 15, "D2": 20}, recibido={}),
        Cliente(id="C5", posicion=(6, 9), prioridad="media", demandas={"D1": 15, "D2": 15}, recibido={}),
        Cliente(id="C6", posicion=(0, 6), prioridad="media", demandas={"D1": 15, "D2": 10}, recibido={})

    ],
    vehiculos=[
        VehiculoEntorno(id="V1", posicion=(4, 4), capacidad_max = 25),
        VehiculoEntorno(id="V2", posicion=(5, 5), capacidad_max = 20),
        VehiculoEntorno(id="V3", posicion=(3, 3), capacidad_max = 30)
    ]
)

In [80]:
# 1. Entorno y Datos
runtime = SingleThreadedAgentRuntime()
# 2. Registro de agentes con corrección de Lambda
for v_data in estado_inicial.vehiculos:
    # Usamos v=v_data para capturar la instancia correcta en cada iteración
    await AgenteVehiculoSimple.register(
        runtime, 
        type=v_data.id, 
        factory=lambda v=v_data: AgenteVehiculoSimple(
            id=v.id, 
            capacidad=v.capacidad_max, 
            coste_paso=v.coste_paso, 
            posicion_inicial=v.posicion,
            penalizacion_t = estado_inicial.penalizacion_trafico
        )
    )

# 3. Ejecución
runtime.start() # [cite: 544]
await ejecutar_simulacion_simple(runtime, estado_inicial)
await runtime.stop() # [cite: 545]
await runtime.close() # [cite: 549]

=== 🚚 SIMULACIÓN LOGÍSTICA - ITERACIÓN 0 ===
[D1]     .        .        .        .        .        [C6]     .        .        .       
.        .        #        #        .        .        .        .        .        .       
.        .        #        .        .        .        [C2]     .        [C3]     .       
.        .        .        V3       .        .        .        .        .        .       
.        .        .        .        V1       T        T        T        .        [C4]    
.        .        .        .        .        V2       .        .        .        .       
.        .        .        .        .        .        .        .        #        [C5]    
.        .        .        [D2]     .        .        .        .        .        #       
.        .        .        .        .        .        .        .        [C1]     .       
.        .        .        .        .        .        .        .        .        .       
----------------------------------------
Cliente C1 (al